# 11 — Ensemble décorrélé (combinaison de modèles)

Le blend cat+lgb sur les MÊMES features n'aidait pas (trop corrélés). Ici on force la
**décorrélation par sous-ensembles de features** :
- **A — full** : toutes les features (te_origin + récence + …) = notre champion (last 0.3607).
- **B — struct** : tout SAUF l'axe taux-du-compte (te_origin, recent_rate) -> forcé sur le
  signal destinataire/structurel -> décorrélé de A.
- **C — xgb** : XGBoost sur full (3e source de diversité).

On blende (rank-average) et on regarde si ça dépasse 0.3607. Boussole : LB ≈ last − 0.004.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything, make_submission
from src.blending import rank_average
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy()
y_all = train[C.TARGET].to_numpy()
folds_full = list(time_folds(train[C.PERIOD]))
try:
    import xgboost as xgb; HAS_XGB = True
except Exception as e:
    HAS_XGB = False; print('XGBoost indisponible -> on l ignore:', e)

In [ ]:
EPS = 1e-6
WINDOWS = (5, 10, 20)
SMOOTHING = 30
# colonnes de l'axe "taux du compte" -> retirées pour le modèle B (struct)
ACCOUNT_RATE_COLS = ["te_origin", "recent_rate_origin_5", "recent_rate_origin_10", "recent_rate_origin_20"]

def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]
    f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)

def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X

def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    rt = recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)
    return pd.concat([X, beh, rec, rt], axis=1)

def feats_train(df, ref):
    X = base_build(df, ref)
    X["te_origin"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SMOOTHING)
    return X

def feats_apply(df, ref):
    X = base_build(df, ref)
    mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SMOOTHING)
    X["te_origin"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm)
    return X

def make_cat():
    from catboost import CatBoostClassifier
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=600, random_seed=42, verbose=False)

def make_xgb():
    import xgboost as xgb
    return xgb.XGBClassifier(objective="binary:logistic", eval_metric="aucpr", max_depth=6,
                             learning_rate=0.05, n_estimators=600, subsample=0.8,
                             colsample_bytree=0.8, random_state=42, n_jobs=-1)

## CV : 3 modèles décorrélés + blends

In [ ]:
oof_A = np.zeros(len(train)); oof_B = np.zeros(len(train)); oof_C = np.zeros(len(train))
for tr_idx, va_idx in folds_full:
    tr_op = tr_idx[op03[tr_idx]]; va_op = va_idx[op03[va_idx]]
    ref = train.iloc[tr_op]
    Xtr = feats_train(train.iloc[tr_op], ref); Xva = feats_apply(train.iloc[va_op], ref)
    yt = y_all[tr_op]
    # A : full
    oof_A[va_op] = make_cat().fit(Xtr, yt).predict_proba(Xva)[:, 1]
    # B : struct (sans l'axe taux du compte)
    Xtr_s = Xtr.drop(columns=ACCOUNT_RATE_COLS); Xva_s = Xva.drop(columns=ACCOUNT_RATE_COLS)
    oof_B[va_op] = make_cat().fit(Xtr_s, yt).predict_proba(Xva_s)[:, 1]
    # C : xgboost full
    if HAS_XGB:
        oof_C[va_op] = make_xgb().fit(Xtr, yt).predict_proba(Xva)[:, 1]
    print("fold ok")

def ap_last(oof):
    return [evaluate_ap(y_all[va_idx[op03[va_idx]]], oof[va_idx[op03[va_idx]]]) for _, va_idx in folds_full]

def show(name, oof):
    pf = ap_last(oof)
    print(f"{name:18s} recent(2) {np.mean(pf[-2:]):.4f} | last {pf[-1]:.4f} | LB~ {pf[-1]-0.004:.4f}")
    return pf

print("--- modèles seuls ---")
show("A full (champion)", oof_A)
show("B struct (no rate)", oof_B)
if HAS_XGB: show("C xgboost full", oof_C)

print("\n--- corrélation des OOF (op_03) ---")
m = op03
print("A-B :", round(np.corrcoef(oof_A[m], oof_B[m])[0, 1], 3),
      "| A-C :", round(np.corrcoef(oof_A[m], oof_C[m])[0, 1], 3) if HAS_XGB else "NA")

print("\n--- blends (rank-average) ---")
def blend_oof(oofs):
    out = np.zeros(len(train))
    for _, va_idx in folds_full:
        vo = va_idx[op03[va_idx]]
        out[vo] = rank_average([o[vo] for o in oofs])
    return out
show("A+B", blend_oof([oof_A, oof_B]))
if HAS_XGB:
    show("A+C", blend_oof([oof_A, oof_C]))
    show("A+B+C", blend_oof([oof_A, oof_B, oof_C]))

## Soumission du meilleur blend (sans calibration) — éditer BEST_OOFS si besoin

In [ ]:
# Choisir la combinaison gagnante d'après la CV ci-dessus :
USE = ["A", "B"]  # ex. ["A"], ["A","B"], ["A","B","C"]

ref_full = train.iloc[np.where(op03)[0]]; yf = y_all[op03]
Xf = feats_train(ref_full, ref_full)
te_op = op03_mask(test).to_numpy(); test_op = test.iloc[np.where(te_op)[0]]
Xte = feats_apply(test_op, ref_full)
Xf_s = Xf.drop(columns=ACCOUNT_RATE_COLS); Xte_s = Xte.drop(columns=ACCOUNT_RATE_COLS)

preds = {}
preds["A"] = make_cat().fit(Xf, yf).predict_proba(Xte)[:, 1]
preds["B"] = make_cat().fit(Xf_s, yf).predict_proba(Xte_s)[:, 1]
if HAS_XGB: preds["C"] = make_xgb().fit(Xf, yf).predict_proba(Xte)[:, 1]

proba = rank_average([preds[k] for k in USE]) if len(USE) > 1 else preds[USE[0]]
full = np.zeros(len(test)); full[te_op] = proba
path = make_submission(test[C.ID], full, "11_ensemble_" + "".join(USE))
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("soumission écrite :", path, "| proba>0 :", int((sub['target'] > 0).sum()))